# 42 — Beam-tier λ sweep (blend, n=2..4, bf16 baseline)

Completes the λ story. Notebook 41 located the greedy-tier knee
(λ* ≈ 0.5–0.6 under the ppl ≤ 2.5 guardrail); the recipe's canonical
λ ≈ 0.4, however, lives at the **beam** tier and was calibrated on
beam-tier AWQ pairs (E1b). This notebook runs the identical seven-point
sweep — λ ∈ {0, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0} — with **true beam search**
(k=4, pool k²=16) and the blend guide over the same 70 nested groups.
All points run fresh (no beam rows exist on these groups). The analysis
overlays notebook 41's greedy curves for the tier comparison and applies
the same two knee criteria per n.

Expectation from notebook 31 (singles): beam's λ response should be far
flatter than greedy's — λ nearly free until some larger threshold.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
LAMBDAS = [0.0, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]
BEAM_BATCH = 4

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

GROUPS33 = json.loads(Path("resources/33_scaling_groups_bf16.json").read_text())
QUADS = json.loads(Path("resources/39_quads.json").read_text())["quads"]
GROUPS = [*GROUPS33["pairs"], *GROUPS33["triples"], *QUADS]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/42_beam_lambda.jsonl")
LOG_PATH.unlink(missing_ok=True)
print(
    f"{len(GROUPS)} groups x {len(LAMBDAS)} lambdas = {len(GROUPS) * len(LAMBDAS)} configs"
)

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
prompts = [chat(u, a) for u, a in QUESTIONS]
harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

used = sorted(
    {s for g in GROUPS for s in (g["a"], g["b"], g.get("c"), g.get("d")) if s}
)
concept = {}
for name in used:
    c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    concept[name] = Concept(synset=c.synset, tensor=c.tensor.to(fur.model.device))
print(f"{len(concept)} concepts built")

for gi, group in enumerate(GROUPS):
    synsets = [s for s in (group["a"], group["b"], group.get("c"), group.get("d")) if s]
    cs = [concept[s] for s in synsets]
    blend_guide = Concept.average(cs)
    for lam in LAMBDAS:
        torch.cuda.empty_cache()
        texts, probe = fur.generate_with_topk_beam_guide(
            prompts,
            concepts=[blend_guide],
            k=K,
            steps=STEPS,
            fluency_weight=lam,
            batch_size=BEAM_BATCH,
        )
        harness.evaluate(
            prompts,
            texts,
            synsets,
            config={
                "model": MODEL_KEY,
                "method": "beam-blend",
                "n": len(synsets),
                "lam": lam,
                **group,
            },
            probe=probe,
            baseline_texts=baseline_texts,
            concepts=dict(zip(synsets, cs)),
            angles=True,
        )
    print(
        f"[{gi + 1}/{len(GROUPS)}] n={len(synsets)} "
        f"{' + '.join(s.split('.')[0] for s in synsets)}: done"
    )

In [ ]:
rows = []
for line in LOG_PATH.read_text().strip().split("\n"):
    r = json.loads(line)
    c = r["config"]
    synsets = [s for s in (c["a"], c["b"], c.get("c"), c.get("d")) if s]
    deltas = [r["expression"][s]["delta"] for s in synsets]
    cosines = [r["angles"][s]["delta"]["token_cos"] for s in synsets]
    rows.append(
        {
            "n": c["n"],
            "lam": c["lam"],
            "joint": min(deltas),
            "joint_cos": min(cosines),
            "all_positive_cos": min(cosines) > 0,
            "ppl_ratio": r["ppl_ratio"],
        }
    )
results = pd.DataFrame(rows)
print(f"gate: {results.groupby('lam').size().to_dict()} records")

summary = (
    results.groupby(["n", "lam"])
    .agg(
        joint=("joint", "mean"),
        joint_cos=("joint_cos", "mean"),
        all_positive_cos=("all_positive_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n_runs=("joint", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/42_beam_lambda_summary.csv", index=False)
summary.round(4)

In [ ]:
greedy = pd.read_csv("resources/41_lambda_sweep_summary.csv")

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
colors = {2: "tab:blue", 3: "tab:orange", 4: "tab:green"}
for ax, (metric, label) in zip(
    axes,
    (
        ("joint_cos", "mean joint token-cosine gain"),
        ("all_positive_cos", "share ALL concepts improved (cosine)"),
        ("ppl_ratio", "mean perplexity ratio"),
    ),
):
    for n in (2, 3, 4):
        sub = summary[summary["n"] == n].sort_values("lam")
        ax.plot(
            sub["lam"], sub[metric], marker="o", color=colors[n], label=f"beam n={n}"
        )
        gsub = greedy[greedy["n"] == n].sort_values("lam")
        ax.plot(
            gsub["lam"], gsub[metric], marker=".", ls=":", color=colors[n], alpha=0.6
        )
    ax.set_xlabel("fluency weight lambda")
    ax.set_ylabel(label)
axes[0].axhline(0, color="gray", lw=0.8)
axes[1].axhline(0.5, color="gray", ls=":", lw=0.8)
axes[2].axhline(2.5, color="gray", ls=":", lw=0.8)
axes[0].legend(fontsize=8)
plt.suptitle("Beam blend lambda-response (solid) vs greedy (dotted), n=2..4")
plt.tight_layout()
plt.savefig("resources/42_beam_lambda_response.png", dpi=150)
plt.show()

print("knee analysis per n (beam tier):")
for n in (2, 3, 4):
    sub = summary[summary["n"] == n].sort_values("lam")
    best = sub.loc[sub["joint_cos"].idxmax()]
    ok = sub[sub["ppl_ratio"] <= 2.5]
    line = (
        f"  n={n}: unconstrained argmax lam={best['lam']:.1f} "
        f"(joint_cos {best['joint_cos']:.4f})"
    )
    if len(ok):
        cons = ok.loc[ok["joint_cos"].idxmax()]
        line += (
            f"; constrained (ppl<=2.5) lam={cons['lam']:.1f} "
            f"(joint_cos {cons['joint_cos']:.4f}, ppl {cons['ppl_ratio']:.2f})"
        )
    else:
        line += "; no lambda meets ppl<=2.5"
    print(line)

print("\nbeam-vs-greedy at matched lambda (joint_cos, all n pooled):")
pooled_b = results.groupby("lam")["joint_cos"].mean()
pooled_g = greedy.groupby("lam").apply(
    lambda s: (s["joint_cos"] * s["n_runs"]).sum() / s["n_runs"].sum()
)
for lam in sorted(set(pooled_b.index) & set(pooled_g.index)):
    print(f"  lam={lam:.1f}: beam {pooled_b[lam]:.4f} vs greedy {pooled_g[lam]:.4f}")